In [ ]:
from retinaface_bboxes import generate_retinaface_bboxes
from deepsort_facetracker import DeepSORT_Tracker
from face_log_generation_utils import save_face_tracklets, generate_csv_from_folder
from tracklet_to_log import generate_log_from_tracklets
import cv2
import warnings
import json
import os, logging
from datetime import datetime
warnings.filterwarnings('ignore')

currentpath = os.getcwd()
logfolder = os.path.join(os.getcwd(),"logs")
os.makedirs(logfolder, exist_ok=True)
entriesfolder = os.path.join(logfolder,"entries")
os.makedirs(entriesfolder, exist_ok=True)
today = datetime.now().strftime("%Y-%m-%d")
folder_path = os.path.join(entriesfolder, today)
os.makedirs(folder_path, exist_ok=True)

log_file = os.path.join(folder_path,"events.log")

logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s'
)

import sqlite3 
db_path = "log.db"

if not os.path.exists(db_path):
	conn = sqlite3.connect("log.db")
	cursor = conn.cursor()

	# Create Log Tablee
	cursor.execute("""
	CREATE TABLE log(
		id INTEGER PRIMARY KEY AUTOINCREMENT,
		filename TEXT,
		timestamp TEXT,
		eventtype text,
		faceid INTEGER)
	""")
	conn.commit()
	conn.close()


conn =sqlite3.connect("log.db")
conn.row_factory = sqlite3.Row


vid = 'test_videos/test_vid.mp4'

In [ ]:
# initalize deepSORT for tracklet generation

tracker = DeepSORT_Tracker()
track_history = {}

In [ ]:
# function to convert retinaface bboxes into deepsort ingestible format

def postprocess_bboxes(bboxes):
    processed_bboxes = []
    for box in bboxes:
        x1, y1, x2, y2, score = box
        w = x2 - x1
        h = y2 - y1

        p_box = [int(x1), int(y1), int(w), int(h)]
        processed_bboxes.append(
            (p_box, float(score))
        )

    return processed_bboxes

In [ ]:
# video inference
def detect_track_faces(vid):
    activetracks = {}
    deletedtracks = {}
    current_tracks_set = set()
    frame_idx = 1
    cap = cv2.VideoCapture(vid)

    persist_set = set()
    while cap.isOpened():
        ret, frame = cap.read()
        if frame is None or cv2.waitKey(1) & 0xFF == 27: 
            break
        else:
            # person detection
            detections = generate_retinaface_bboxes(frame)
            deepsort_bboxes = postprocess_bboxes(detections)
            logging.info(f"Recieved RetinaFace detections {deepsort_bboxes} for frame no: {frame_idx}")
            frame_idx+=1


            # person tracking
            tracks_current = tracker.object_tracker.update_tracks(deepsort_bboxes, frame = frame)
            for track in tracks_current:
                persist_set.add(track)


            logging.info(f"Started tracklets for {len(tracks_current)} tracks.")
            tracker.display_track(track_history, tracks_current, frame)
            
            # tracklet generation
            for track in tracks_current:
                img_name, timestamp, track_id,status = save_face_tracklets(frame, track)

                if status =="Y":
                    if int(track_id) not in activetracks.keys():
                        print(f'Trackid {int(track_id)} does not exist - Hence entrying')
                        activetracks[int(track_id)] = int(track_id)
                        cursor = conn.cursor()
                        cursor.execute("""
                            INSERT INTO log (filename,timestamp,eventtype,faceid) 
                            values (?,?,?,?)
                            """, (img_name,timestamp,"entry",track_id))
                        conn.commit()
                        logging.info(f"Entry logged for tracked - {track_id}.")
                    else:
                        print(f'trackid {int(track_id)} already exists')

                    '''                if status =="Y":
                    print(f"Current Tracks: {current_tracks_set}")
                    #dead_tracks = set(activetracks.keys()) - set(current_tracks_set)
                    actualdeadtracks =  set(deletedtracks.keys()) - set(activetracks.keys())
                    print(f"Dead Track: {actualdeadtracks}")
                    for i in actualdeadtracks:
                        print(f"Active Tracks: {activetracks.keys()}")
                        print(f"Deleted Tracks: {deletedtracks.keys()}")
                        deletedtracks[i]=i
                        cursor = conn.cursor()
                        cursor.execute("""
                            INSERT INTO log (filename,timestamp,eventtype,faceid) 
                            values (?,?,?,?)
                            """, (img_name,timestamp,"exit",i))
                        conn.commit()
                        logging.info(f"Exit logged for tracked - {track_id}.")
                    
                    pop_track=[]
                    for p_track in persist_set:
                        if (p_track.time_since_update >3) and (p_track.track_id in activetracks.keys()):
                            cursor = conn.cursor()
                            cursor.execute("""
                            INSERT INTO log (filename,timestamp,eventtype,faceid) 
                            values (?,?,?,?)
                            """, (img_name,timestamp,"exit", p_track.track_id))
                            conn.commit()
                            logging.info(f"Exit logged for tracked - {track_id}.")
                            pop_track.append(p_track)
                    for i in pop_track:
                        persist_set.discard(i)'''


            # resize frame before display
            resized_frame = cv2.resize(frame, None, fx=0.75, fy=0.75, interpolation=cv2.INTER_AREA)
            cv2.imshow('frame', resized_frame)

    cap.release()
    cv2.destroyAllWindows()

In [ ]:
# generate tracklets
detect_track_faces(vid)

In [ ]:
# generate unique faces
tracklet_dir = 'face_tracklets/' 
generate_log_from_tracklets(tracklet_dir)

In [ ]:
# generate csv report
unique_faces_path = 'unique_faces/'
generate_csv_from_folder(unique_faces_path)